# 🎨 AI Image Generation with Hugging Face + Google Colab
## Complete A–Z Practical

This notebook teaches **Text-to-Image** and **Image-to-Image** generation using Hugging Face Diffusers in Google Colab.

### Learning goals
- Explain Generative AI and diffusion models.
- Generate images from text prompts.
- Understand prompts, negative prompts, seeds, steps and guidance scale.
- Upload an existing image and transform it.
- Understand the roles of the text encoder, U-Net, scheduler and VAE.
- Experiment with image-to-image strength.
- Use AI image generation responsibly.

# 1. What is Generative AI?

Generative AI creates new content from input.

Examples: Text → Text, Text → Image, Image → Image, Text → Audio, Text → Video, Text → Code.

For this practical we use **Google Colab + Hugging Face Diffusers + Stable Diffusion + PyTorch**.

# 2. How AI Creates an Image

```text
User Prompt
     ↓
Tokenizer / Text Encoder
     ↓
Text representation
     ↓
Random Noise
     ↓
Repeated denoising guided by the prompt
     ↓
Latent image
     ↓
VAE Decoder
     ↓
Final Image
```

The model learns relationships between language and visual patterns during training.

# 3. Diffusion in Simple English

### During training
```text
Real Image → add noise → noisy image
                     ↓
             model learns denoising
```

### During generation
```text
Random Noise → denoise → denoise → denoise → Final Image
```

The prompt guides the denoising process. This is a simplified teaching model; modern diffusion systems operate in a learned latent space and use several components.

# 4. Text-to-Image vs Image-to-Image

### Text-to-Image
```text
Text prompt → AI → New image
```

### Image-to-Image
```text
Existing image + prompt → AI → Transformed image
```

The existing image supplies visual information, while the prompt describes the desired transformation.

# 5. Enable GPU in Google Colab

Select **Runtime → Change runtime type → T4 GPU** (if available).

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable a GPU runtime.")

# 6. Install Libraries

We use `diffusers`, `transformers`, `accelerate`, `safetensors`, `huggingface_hub`, and `Pillow`.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub pillow

# 7. Import Libraries

In [ ]:
import torch
from PIL import Image
from IPython.display import display
from diffusers import StableDiffusionPipeline

# 8. Hugging Face Access

Model used: `runwayml/stable-diffusion-v1-5`.

If Hugging Face requests authentication or model terms, log in and follow the model access instructions.

In [ ]:
# Optional authentication
from huggingface_hub import login
# login()

# 9. Load Stable Diffusion

FP16 is used on CUDA to reduce memory usage. The first run downloads the model.

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe = pipe.to("cuda")
print("Model loaded successfully!")

# 10. What Is Inside the Pipeline?

```text
Text Prompt
    ↓
Tokenizer → Text Encoder
    ↓
Text Embeddings
    ↓
Random Noise → U-Net Denoiser ← Scheduler
    ↓
Latent Image
    ↓
VAE Decoder
    ↓
Final Image
```

**Tokenizer:** breaks text into tokens.
**Text encoder:** converts text into numerical representation.
**U-Net:** predicts denoising information.
**Scheduler:** controls denoising steps.
**VAE:** converts between image space and latent space.

# 11. First AI-Generated Image 🚀

In [ ]:
prompt = "A friendly robot teaching programming to university students in a modern classroom"

image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    height=512,
    width=512
).images[0]

display(image)

# 12. Understanding an Image Prompt

A useful prompt can describe: subject, action, environment, style, lighting, composition and detail.

Example:
```text
A young programmer, working on a laptop, inside a futuristic classroom,
realistic photography, cinematic lighting, wide composition, highly detailed
```

# 13. Prompt Experiment

In [ ]:
prompts = [
    "A futuristic city at sunset, cinematic digital art",
    "A cute robot reading a programming book in a library, 3D illustration",
    "A traditional Sri Lankan village landscape, realistic photography",
    "A cyberpunk programmer working at night, neon lights, cinematic"
]

for p in prompts:
    print("\nPROMPT:", p)
    result = pipe(prompt=p, num_inference_steps=25, guidance_scale=7.5, height=512, width=512).images[0]
    display(result)

# 14. Negative Prompts

A negative prompt describes things we prefer not to generate. Example: `blurry, low quality, distorted, extra fingers, watermark, text`.

Negative prompts are guidance, not guaranteed exclusions.

In [ ]:
prompt = "A professional software engineer working in a modern technology office, realistic photography, detailed"
negative_prompt = "blurry, low quality, distorted face, extra fingers, bad anatomy, watermark, text"

image = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=30, guidance_scale=7.5, height=512, width=512).images[0]
display(image)

# 15. Seeds and Reproducibility

A **seed** controls the random starting noise. Same prompt + settings + seed can reproduce a result. Changing the seed changes the starting point.

In [ ]:
seed = 42
generator = torch.Generator(device="cuda").manual_seed(seed)

prompt = "A futuristic university campus in the year 2050, cinematic digital art"
image = pipe(prompt=prompt, generator=generator, num_inference_steps=30, guidance_scale=7.5, height=512, width=512).images[0]
display(image)

# 16. Important Parameters

- `num_inference_steps`: number of denoising steps.
- `guidance_scale`: how strongly generation follows the prompt.
- `height`, `width`: output dimensions.
- `seed`: controls the random starting point.

# 17. Student Prompt Activity

Keep the same seed and progressively improve:

A: `A student learning Python`

B: `A university student learning Python in a modern computer laboratory`

C: `A university student learning Python in a modern computer laboratory, cinematic lighting, realistic photography, highly detailed`

Discuss what visual details change.

# 18. Save the Image

In [ ]:
output_path = "/content/generated_image.png"
image.save(output_path)
print("Saved:", output_path)

# 19. Download the Image

In [ ]:
from google.colab import files
files.download("/content/generated_image.png")

# 20. Image-to-Image 🔄

Text-to-image starts from random noise. Image-to-image starts from an existing image:

```text
Existing Image → VAE Encoder → Latent + controlled noise
                                      ↓
                              Denoising + prompt
                                      ↓
                                  VAE Decoder
                                      ↓
                                  New Image
```

Low strength generally preserves more of the original. High strength generally allows larger changes.

# 21. Upload an Existing Image

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
filename = next(iter(uploaded))
input_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
input_image = input_image.resize((512, 512))
print("Uploaded:", filename)
display(input_image)

# 22. Load the Image-to-Image Pipeline

We release the previous pipeline first to reduce GPU memory usage.

In [ ]:
import gc
del pipe
gc.collect()
torch.cuda.empty_cache()

from diffusers import StableDiffusionImg2ImgPipeline
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, use_safetensors=True
)
img2img_pipe = img2img_pipe.to("cuda")
print("Image-to-image pipeline loaded.")

# 23. Transform the Existing Image

In [ ]:
prompt = "A futuristic cinematic version of the scene, modern technology, dramatic lighting, highly detailed"
negative_prompt = "blurry, low quality, distorted, watermark, text"

result = img2img_pipe(
    prompt=prompt, negative_prompt=negative_prompt, image=input_image,
    strength=0.55, guidance_scale=7.5, num_inference_steps=30
).images[0]
display(result)

# 24. Experiment with `strength`

Try `0.25`, `0.50`, and `0.75`. Lower strength generally preserves more of the original; higher strength generally makes larger changes.

In [ ]:
strength_values = [0.25, 0.50, 0.75]
prompt = "A futuristic cinematic version of the same scene, advanced technology, dramatic lighting, detailed"

for strength in strength_values:
    print(f"Strength = {strength}")
    result = img2img_pipe(prompt=prompt, image=input_image, strength=strength, guidance_scale=7.5, num_inference_steps=30).images[0]
    display(result)

# 25. How Existing Image Regeneration Happens

```text
Existing Image → VAE Encoder → Latent Image
                               ↓
                         Controlled Noise
                               ↓
                      U-Net denoising ← Text embedding
                               ↓
                        VAE Decoder
                               ↓
                           New Image
```

The result is influenced by the source image, prompt, strength, seed/randomness, steps and guidance scale.

# 26. Text-to-Image vs Image-to-Image

| Feature | Text-to-Image | Image-to-Image |
|---|---|---|
| Input | Text | Text + existing image |
| Starting point | Random noise | Existing image + noise |
| Original preserved? | No | Partially |
| Main control | Prompt | Prompt + strength |
| Use | Create new artwork | Transform an image |

# 27. Advanced Topic: Inpainting

Inpainting regenerates selected areas:

```text
Original image + Mask + Prompt → Regenerate selected region
```

Examples: replace a background, change an object, remove an unwanted area, add an object, or repair part of an image.

# 28. How Does the AI Know What a Cat Looks Like?

During training, the model learns relationships between language and visual patterns. It can learn associations involving words such as `cat`, `kitten`, `fur`, `ears`, `eyes`, `whiskers`, and `animal`. The learned representation is much more complex than a hand-written drawing rule.

# 29. Is the AI Just Copying Existing Images?

The basic generation process is not `Prompt → internet search → copy image`. A diffusion model generates through learned parameters and iterative denoising.

However, training-data issues matter. Outputs can sometimes resemble training examples, so copyright, licensing, consent, attribution and bias should be considered.

# 30. CUDA Out-of-Memory Troubleshooting

If you see `CUDA out of memory`: use 512×512, restart the runtime, avoid multiple pipelines, reduce steps, and clean memory.

In [ ]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory cleanup requested.")

# 31. Common Problems

### Hugging Face 401/403
Authentication or model-access terms may be required.

### Slow generation
Use a GPU, 512×512 images and 20–30 steps.

### Poor output
Experiment with prompt detail, negative prompt, seed, steps, guidance scale and image-to-image strength.

# 32. Prompt Engineering Formula

```text
[Subject] + [Action] + [Environment] + [Style] + [Lighting] + [Composition] + [Detail]
```

Example:
`A young software engineering student, writing Python code, inside a modern university computer lab, realistic photography, soft cinematic lighting, medium shot, highly detailed`

# 33. Student Activity — Build Your Own Image

Create an image for a fictional technology company. Include subject, environment, style, lighting, composition and at least 3 descriptive details.

Submit the prompt, generated image, seed, number of steps, guidance scale and a short explanation.

# 34. Student Activity — Image-to-Image

Upload an image and create versions with strength `0.25`, `0.50`, and `0.75`. Use the same prompt.

Answer: Which is most similar? Which changed most? What does strength control? What role does the prompt play?

# 35. Mini Project — AI Creative Studio

Build a Colab project that can generate from text, accept an uploaded image, transform it, allow prompt/negative prompt/seed/strength/steps to be changed, and save the result.

Suggested interface:
```text
Prompt:          [________________]
Negative Prompt: [________________]
Strength:        [0.50]
Steps:           [30]
Seed:            [42]
[Generate Image]
```

# 36. Responsible AI

- Get permission before using someone's personal image.
- Do not create deceptive impersonation content.
- Do not present AI-generated images as real when that would mislead people.
- Respect copyright and licensing.
- Consider bias in generated content.
- Be careful with sensitive personal images.
- Label AI-generated content when appropriate.

# 37. Knowledge Check

1. What is Generative AI?
2. What is the basic idea behind diffusion?
3. What does the text encoder do?
4. What is the role of the U-Net?
5. What does the scheduler control?
6. What is the purpose of the VAE?
7. What is a seed?
8. What does guidance scale influence?
9. What does image-to-image strength control?
10. What is the difference between text-to-image and image-to-image?

# 38. Final A–Z Summary

```text
A → AI / Generative AI
B → Background and composition
C → Colab
D → Diffusion
E → Encoder
F → Fine visual details
G → Guidance scale
H → Hugging Face
I → Image-to-image
J → JSON/configuration concepts
K → Knowledge learned during training
L → Latent space
M → Model
N → Negative prompt
O → Output image
P → Prompt
Q → Quality
R → Random noise
S → Seed / Strength / Scheduler
T → Text-to-image
U → U-Net
V → VAE
W → Weights
X → eXperimentation
Y → Your prompt
Z → Zero-shot generation idea
```

### Key sentence for students
> **A diffusion image model starts from noise, or a noisy version of an existing image, and repeatedly denoises it while using the text prompt as guidance to produce an image.**